# Taller MapReduce con Python puro: solución de ejercicios

**Autor:** Sebastián Velasco Ardila
**Programa:** Maestría en Ciencia de Datos, Universidad Pontificia Bolivariana
**Curso:** [nombre del curso] · **Docente:** [nombre] · **Fecha:** octubre de 2026

## Contenido

| Sección | Ejercicios |
|---|---|
| Nivel 1: fundamentos | 1.1 a 1.5 |
| Nivel 2: procesamiento de archivos | 2.1 a 2.3 |
| Nivel 3: retos avanzados | 3.1 a 3.4 |
| Rendimiento | P.1 y P.2 |

Cada ejercicio incluye el objetivo, la solución con mapper y reducer documentados, la prueba del caso borde y las observaciones sobre el resultado.

## Cómo ejecutarlo

- El notebook debe estar en `modules/01-mapreduce/` del repositorio del curso, porque usa rutas relativas al framework (`01-pure-python/01-basics/`) y a los datos (`datasets/`).
- Requiere Python 3 sin librerías adicionales. Se ejecutó con Python 3.13 en Windows.
- Las primeras celdas de código son de preparación: ubican el framework y muestran su contenido antes de usarlo.

## Hallazgos principales

1. **Ejercicio 3.4:** la regla de 2 desviaciones estándar no detecta la anomalía con los datos del enunciado. Con 3 lecturas es matemáticamente imposible que se active; se requieren al menos 6.
2. **Ejercicio P.1:** para un archivo de 0,75 MB, la ejecución paralela es más lenta que la secuencial, porque la sobrecarga de coordinación supera al trabajo útil.
3. **Ejercicio P.2:** los bloques muy pequeños penalizan el rendimiento por el costo fijo de cada bloque.
4. Varias salidas esperadas del enunciado (1.1, 2.2 y 2.3) son ilustrativas y no coinciden con el resultado real de los datos; cada caso está verificado y anotado.

In [26]:
from pathlib import Path

BASE = Path("01-pure-python")

print(Path.cwd())
print((BASE / "01-basics" / "mapreduce_framework.py").read_text(encoding="utf-8"))

c:\Users\velas\OneDrive\Documentos\Personal\Maestria UPB\BIGDATA MS\bigdata-101-main (1)\bigdata-101-main\modules\01-mapreduce
"""
Basic MapReduce framework in pure Python.

This module provides a simple MapReduce implementation
to process data locally and understand the paradigm.
"""

from collections import defaultdict
from typing import Callable, Iterator, Tuple, Any, List


def mapreduce(
    data: List[Any],
    mapper: Callable[[Any], Iterator[Tuple[Any, Any]]],
    reducer: Callable[[Any, List[Any]], Any]
) -> dict:
    """
    Executes a complete MapReduce process.
    
    Args:
        data: List of items to process
        mapper: Function that transforms each item into (key, value) pairs
        reducer: Function that aggregates values by key
    
    Returns:
        Dictionary with final results {key: reduced_value}
    """
    # MAP phase
    mapped = []
    for item in data:
        mapped.extend(mapper(item))
    
    # SHUFFLE/SORT phase
    shuffled = defaultdict(lis

In [27]:
import sys

sys.path.append(str(BASE / "01-basics"))
from mapreduce_framework import mapreduce

In [28]:
texto = [
    "MapReduce is a programming model",
    "for processing large data sets",
]

def mapper(linea):
    """Emite un par (letra, 1) por cada letra de la línea, en minúscula."""
    for caracter in linea.lower():
        if caracter.isalpha():
            yield (caracter, 1)

def reducer(letra, conteos):
    """Suma las apariciones de una letra."""
    return sum(conteos)

resultado = mapreduce(texto, mapper, reducer)
print(dict(sorted(resultado.items())))

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

{'a': 6, 'c': 2, 'd': 3, 'e': 6, 'f': 1, 'g': 4, 'i': 3, 'l': 2, 'm': 4, 'n': 2, 'o': 4, 'p': 3, 'r': 6, 's': 5, 't': 2, 'u': 1}
{}


## Nivel 1 · Ejercicio 1.1: Contador de caracteres

**Objetivo:** contar la frecuencia de cada letra, solo letras y sin distinguir mayúsculas de minúsculas.

**Solución:** el mapper normaliza la línea a minúsculas y emite un par `(letra, 1)` por cada carácter alfabético; el reducer suma los unos de cada letra.

**Nota de verificación:** el enunciado muestra `'a': 5` y `'e': 4` como salida esperada, pero el conteo manual sobre las dos líneas da 6 apariciones para cada una, que es lo que devuelve la ejecución. La salida del enunciado es ilustrativa, no exacta.

In [29]:
print((BASE / "01-basics" / "wordcount.py").read_text(encoding="utf-8"))

"""
WordCount - Classic MapReduce example.

Counts the frequency of each word in a text.
"""

from mapreduce_framework import mapreduce, print_results


def mapper(line):
    """
    Transforms a text line into (word, 1) pairs.
    
    Args:
        line: Text line
    
    Yields:
        Tuples (word, 1) for each word in the line
    """
    words = line.lower().split()
    for word in words:
        # Clean basic punctuation
        word = word.strip('.,!?;:"()[]{}')
        if word:
            yield (word, 1)


def reducer(key, values):
    """
    Sums all counts for a word.
    
    Args:
        key: The word
        values: List of counts (all are 1)
    
    Returns:
        Total occurrences of the word
    """
    return sum(values)


if __name__ == "__main__":
    # Sample data
    text = [
        "MapReduce is a programming model",
        "MapReduce processes large volumes of data",
        "The MapReduce model has two main phases",
        "The Map phase transforms th

In [30]:
codigo = (BASE / "01-basics" / "wordcount.py").read_text(encoding="utf-8")
print(codigo[codigo.find("def reducer"):])

def reducer(key, values):
    """
    Sums all counts for a word.
    
    Args:
        key: The word
        values: List of counts (all are 1)
    
    Returns:
        Total occurrences of the word
    """
    return sum(values)


if __name__ == "__main__":
    # Sample data
    text = [
        "MapReduce is a programming model",
        "MapReduce processes large volumes of data",
        "The MapReduce model has two main phases",
        "The Map phase transforms the data",
        "The Reduce phase aggregates the results"
    ]
    
    print("Input text:")
    for line in text:
        print(f"  {line}")
    
    # Execute MapReduce
    results = mapreduce(text, mapper, reducer)
    
    # Show results
    print_results(results, "Word Count", limit=10)
    
    print(f"Total unique words: {len(results)}")
    print(f"Total words: {sum(results.values())}")



In [31]:
texto = [
    "MapReduce is a programming model",
    "MapReduce processes large volumes of data",
    "The MapReduce model has two main phases",
    "The Map phase transforms the data",
    "The Reduce phase aggregates the results",
]

def mapper(linea):
    """Emite (palabra, 1) solo para las palabras de más de 5 caracteres."""
    for palabra in linea.lower().split():
        palabra = palabra.strip('.,!?;:"()[]{}')
        if len(palabra) > 5:
            yield (palabra, 1)

def reducer(palabra, conteos):
    """Suma las apariciones de una palabra."""
    return sum(conteos)

resultado = mapreduce(texto, mapper, reducer)
for palabra, total in sorted(resultado.items()):
    print(f"{palabra}: {total}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

aggregates: 1
mapreduce: 3
phases: 1
processes: 1
programming: 1
reduce: 1
results: 1
transforms: 1
volumes: 1
{}


## Ejercicio 1.2: Palabras largas

**Objetivo:** contar únicamente las palabras de más de 5 caracteres.

**Solución:** se parte del WordCount original y se cambia la condición del mapper: después de limpiar la puntuación, solo se emite el par si `len(palabra) > 5`. El reducer no cambia.

**Observaciones:** la limpieza debe ir antes de medir la longitud, o la puntuación pegada inflaría el conteo de caracteres. La condición es estricta, por lo que `model`, `large` y `phase` (5 letras) quedan excluidas. Al filtrar en el mapper, las palabras cortas nunca llegan al shuffle ni al reducer.

In [32]:
ventas = [
    {'product': 'Laptop', 'amount': 1200},
    {'product': 'Mouse', 'amount': 25},
    {'product': 'Laptop', 'amount': 1100},
    {'product': 'Mouse', 'amount': 30},
    {'product': 'Laptop', 'amount': 1250},
    {'product': 'Keyboard', 'amount': 75},
    {'product': 'Keyboard', 'amount': 80},
]

def mapper(venta):
    """Emite (producto, monto) por cada venta."""
    yield (venta['product'], venta['amount'])

def reducer(producto, montos):
    """Calcula el monto promedio de un producto, con un decimal."""
    return round(sum(montos) / len(montos), 1)

resultado = mapreduce(ventas, mapper, reducer)
for producto, promedio in resultado.items():
    print(f"{producto}: {promedio}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

Laptop: 1183.3
Mouse: 27.5
Keyboard: 77.5
{}


## Ejercicio 1.3: Promedio de ventas por producto

**Objetivo:** calcular el monto promedio de venta de cada producto.

**Solución:** el mapper emite el par `(producto, monto)` por cada venta. El reducer recibe todos los montos de un producto y devuelve su promedio redondeado a un decimal.

**Observaciones:** a diferencia de los conteos, aquí el valor emitido es el dato real y no un 1. No existe riesgo de división por cero, porque el framework solo invoca al reducer para claves que recibieron al menos un valor; con una entrada vacía el resultado es un diccionario vacío.

In [33]:
codigo = (BASE / "01-basics" / "examples" / "temperatures.py").read_text(encoding="utf-8")
print(codigo[codigo.find("def mapper"):])

def mapper(record):
    """
    Extracts (city, temperature) from each record.
    
    Args:
        record: Dictionary with city, temperature, and date
    
    Yields:
        Tuple (city, temperature)
    """
    yield (record['city'], record['temperature'])


def reducer(city, temperatures):
    """
    Calculates average temperature for a city.
    
    Args:
        city: City name
        temperatures: List of temperature values
    
    Returns:
        Average temperature rounded to 1 decimal
    """
    return round(sum(temperatures) / len(temperatures), 1)


if __name__ == "__main__":
    # Temperature data from Colombian cities
    temperatures = [
        {'city': 'Medellin', 'temperature': 22, 'date': '2026-01-01'},
        {'city': 'Bogota', 'temperature': 14, 'date': '2026-01-01'},
        {'city': 'Medellin', 'temperature': 24, 'date': '2026-01-02'},
        {'city': 'Cali', 'temperature': 28, 'date': '2026-01-01'},
        {'city': 'Bogota', 'temperature': 13, 'date'

In [34]:
inicio = codigo.find("if __name__")
print("\n".join(codigo[inicio:].splitlines()[:28]))

if __name__ == "__main__":
    # Temperature data from Colombian cities
    temperatures = [
        {'city': 'Medellin', 'temperature': 22, 'date': '2026-01-01'},
        {'city': 'Bogota', 'temperature': 14, 'date': '2026-01-01'},
        {'city': 'Medellin', 'temperature': 24, 'date': '2026-01-02'},
        {'city': 'Cali', 'temperature': 28, 'date': '2026-01-01'},
        {'city': 'Bogota', 'temperature': 13, 'date': '2026-01-02'},
        {'city': 'Cali', 'temperature': 30, 'date': '2026-01-02'},
        {'city': 'Medellin', 'temperature': 23, 'date': '2026-01-03'},
        {'city': 'Bogota', 'temperature': 15, 'date': '2026-01-03'},
        {'city': 'Cartagena', 'temperature': 32, 'date': '2026-01-01'},
        {'city': 'Cartagena', 'temperature': 33, 'date': '2026-01-02'},
    ]
    
    print(f"Analyzing {len(temperatures)} temperature records...\n")
    
    # Execute MapReduce
    results = mapreduce(temperatures, mapper, reducer)
    
    # Show results
    print("=" * 40)
 

In [35]:
temperaturas = [
    {'city': 'Medellin', 'temperature': 22, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 14, 'date': '2026-01-01'},
    {'city': 'Medellin', 'temperature': 24, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 28, 'date': '2026-01-01'},
    {'city': 'Bogota', 'temperature': 13, 'date': '2026-01-02'},
    {'city': 'Cali', 'temperature': 30, 'date': '2026-01-02'},
    {'city': 'Medellin', 'temperature': 23, 'date': '2026-01-03'},
    {'city': 'Bogota', 'temperature': 15, 'date': '2026-01-03'},
    {'city': 'Cartagena', 'temperature': 32, 'date': '2026-01-01'},
    {'city': 'Cartagena', 'temperature': 33, 'date': '2026-01-02'},
]

def mapper(registro):
    """Emite (ciudad, temperatura) por cada registro."""
    yield (registro['city'], registro['temperature'])

def reducer(ciudad, temps):
    """Devuelve el mínimo, el máximo y el promedio de temperatura de una ciudad."""
    return {
        'min': min(temps),
        'max': max(temps),
        'avg': round(sum(temps) / len(temps), 1),
    }

resultado = mapreduce(temperaturas, mapper, reducer)
for ciudad, metricas in resultado.items():
    print(f"{ciudad}: {metricas}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

Medellin: {'min': 22, 'max': 24, 'avg': 23.0}
Bogota: {'min': 13, 'max': 15, 'avg': 14.0}
Cali: {'min': 28, 'max': 30, 'avg': 29.0}
Cartagena: {'min': 32, 'max': 33, 'avg': 32.5}
{}


## Ejercicio 1.4: Estadísticas de temperatura

**Objetivo:** calcular la temperatura mínima, máxima y promedio de cada ciudad.

**Solución:** el mapper emite `(ciudad, temperatura)` y descarta la fecha, que no interviene en el cálculo. El reducer recibe la lista de temperaturas de una ciudad y devuelve un diccionario con las tres métricas.

**Observaciones:** un reducer puede devolver una estructura compuesta y no solo un escalar, lo que permite obtener varias agregaciones en una sola pasada sobre los datos. El enunciado muestra solo Medellín y Bogotá; la ejecución incluye también Cali y Cartagena, presentes en los datos de origen.

In [36]:
ventas = [
    {'product': 'Laptop', 'category': 'Electronics', 'amount': 1200},
    {'product': 'Mouse', 'category': 'Electronics', 'amount': 25},
    {'product': 'Desk', 'category': 'Furniture', 'amount': 600},
    {'product': 'Chair', 'category': 'Furniture', 'amount': 350},
    {'product': 'Monitor', 'category': 'Electronics', 'amount': 450},
]

def mapper(venta):
    """Emite (categoría, monto) por cada venta."""
    yield (venta['category'], venta['amount'])

def reducer(categoria, montos):
    """Devuelve la cantidad de ventas, el total y el promedio de una categoría."""
    return {
        'count': len(montos),   # cuántas ventas hay en la lista
        'total': sum(montos),   # la suma de los montos
        'avg': round(sum(montos) / len(montos), 1),     # el promedio, con un decimal
    }

resultado = mapreduce(ventas, mapper, reducer)
for categoria, metricas in resultado.items():
    print(f"{categoria}: {metricas}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

Electronics: {'count': 3, 'total': 1675, 'avg': 558.3}
Furniture: {'count': 2, 'total': 950, 'avg': 475.0}
{}


## Ejercicio 1.5: Conteo por categoría

**Objetivo:** contar las ventas de cada categoría y calcular su total y su promedio.

**Solución:** el mapper cambia la clave de agrupación de producto a categoría y emite `(categoría, monto)`. El reducer devuelve un diccionario con la cantidad de ventas (`len`), el total (`sum`) y el promedio redondeado a un decimal.

**Observaciones:** la clave que emite el mapper define el nivel de agregación del resultado. Con los mismos datos de entrada, cambiar `product` por `category` basta para pasar de un análisis por producto a uno por categoría, sin tocar el framework.

In [37]:
DATOS = Path("../../datasets/mapreduce")

print(sorted(p.name for p in DATOS.iterdir()))

lineas = (DATOS / "sample_text.txt").read_text(encoding="utf-8").splitlines()
print(len(lineas), "líneas")
print(lineas[:3])

['README.md', 'index', 'sample_bigdata.txt', 'sample_text.txt']
10 líneas
['MapReduce is a programming model and an associated implementation for processing and generating large data sets.', 'Users specify a map function that processes a key value pair to generate a set of intermediate key value pairs.', 'A reduce function merges all intermediate values associated with the same intermediate key.']


In [38]:
lineas = (DATOS / "sample_text.txt").read_text(encoding="utf-8").splitlines()

def mapper(linea):
    """Emite (longitud, 1) por cada palabra de la línea."""
    for palabra in linea.split():
        palabra = palabra.strip('.,!?;:"()[]{}')
        if palabra:
            yield (len(palabra), 1)   

def reducer(longitud, conteos):
    """Suma cuántas palabras tienen esa longitud."""
    return sum(conteos)

resultado = mapreduce(lineas, mapper, reducer)
for longitud, total in sorted(resultado.items()):
    print(f"{longitud}: {total} words")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

1: 10 words
2: 20 words
3: 33 words
4: 17 words
5: 14 words
6: 13 words
7: 12 words
8: 17 words
9: 8 words
10: 12 words
11: 8 words
12: 7 words
13: 3 words
14: 1 words
{}


## Nivel 2 · Ejercicio 2.1: Distribución de longitud de palabras

**Objetivo:** contar cuántas palabras hay de cada longitud en `sample_text.txt`.

**Solución:** el archivo se lee como una lista de líneas. El mapper limpia la puntuación de cada palabra y emite `(longitud, 1)`; el reducer suma los unos de cada longitud.

**Observaciones:** la clave no tiene que ser un dato presente en la entrada, puede ser un valor derivado. Aquí se agrupa por `len(palabra)`. El archivo contiene 175 palabras; la longitud más frecuente es 3 letras (33 palabras) y la palabra más larga tiene 14.

In [39]:
datos_archivos = [
    (archivo.name, linea)
    for archivo in sorted(DATOS.glob("*.txt"))
    for linea in archivo.read_text(encoding="utf-8").splitlines()
]
print(len(datos_archivos), "registros. Ejemplo:", datos_archivos[0][0])

def mapper(dato):
    """Emite (archivo, palabra) por cada palabra de la línea."""
    archivo, linea = dato
    for palabra in linea.lower().split():
        palabra = palabra.strip('.,!?;:"()[]{}')
        if palabra:
            yield (archivo, palabra)

def reducer(archivo, palabras):
    """Cuenta las palabras distintas del archivo."""
    return len(set(palabras))   # ¿cuántas palabras DISTINTAS hay en la lista?

resultado = mapreduce(datos_archivos, mapper, reducer)
for archivo, unicas in resultado.items():
    print(f"{archivo}: {unicas} unique words")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

18 registros. Ejemplo: sample_bigdata.txt
sample_bigdata.txt: 79 unique words
sample_text.txt: 116 unique words
{}


## Ejercicio 2.2: Palabras únicas por archivo

**Objetivo:** determinar cuántas palabras distintas contiene cada archivo de `datasets/mapreduce/`.

**Solución:** la entrada se construye como una lista de parejas `(archivo, línea)`, de modo que cada registro conserva su origen. El mapper emite `(archivo, palabra)` y el reducer elimina los duplicados con `set` y cuenta los elementos restantes.

**Observaciones:** se procesaron 18 registros de dos archivos. `sample_bigdata.txt` tiene 79 palabras únicas y `sample_text.txt` tiene 116. Los valores del enunciado (85 y 120) son ilustrativos. El conteo depende de las reglas de normalización: aquí se pasa todo a minúsculas y se retira la puntuación de los extremos.

In [40]:
def mapper(dato):
    """Emite la pareja del 2.2 invertida, para agrupar por palabra."""
    archivo, linea = dato
    for palabra in linea.lower().split():
        palabra = palabra.strip('.,!?;:"()[]{}')
        if palabra:
            yield (palabra, archivo)   

def reducer(palabra, archivos):
    """Devuelve la lista ordenada y sin repetidos de archivos que contienen la palabra."""
    return sorted(set(archivos))

indice = mapreduce(datos_archivos, mapper, reducer)

print(len(indice), "palabras en el índice")
en_ambos = [p for p, a in indice.items() if len(a) > 1]
print(len(en_ambos), "palabras aparecen en los dos archivos\n")

for palabra in ["mapreduce", "hadoop", "data"]:
    print(f"{palabra}: {indice.get(palabra, [])}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

177 palabras en el índice
18 palabras aparecen en los dos archivos

mapreduce: ['sample_text.txt']
hadoop: ['sample_bigdata.txt']
data: ['sample_bigdata.txt', 'sample_text.txt']
{}


## Ejercicio 2.3: Índice invertido

**Objetivo:** construir, para cada palabra, la lista de archivos que la contienen.

**Solución:** el mapper invierte la pareja del ejercicio anterior y emite `(palabra, archivo)`, con lo que el framework agrupa por palabra. El reducer elimina los archivos repetidos y devuelve la lista ordenada.

**Observaciones:** el índice contiene 177 palabras, de las cuales 18 aparecen en los dos archivos. La cifra es consistente con el ejercicio 2.2: 79 + 116 palabras únicas menos las 18 compartidas. En estos datos, `mapreduce` solo aparece en `sample_text.txt`, a diferencia de lo que muestra el enunciado, cuya salida es ilustrativa.

In [41]:
def mapper_palabras(linea):
    """Emite (palabra, 1) por cada palabra de la línea."""
    for palabra in linea.lower().split():
        palabra = palabra.strip('.,!?;:"()[]{}')
        if palabra:
            yield (palabra, 1)

def reducer_suma(palabra, conteos):
    """Suma las apariciones de una palabra."""
    return sum(conteos)

def top_n_palabras(texto, n=10):
    """Devuelve las n palabras más frecuentes como lista de (palabra, conteo)."""
    conteos = mapreduce(texto, mapper_palabras, reducer_suma)
    ordenadas = sorted(conteos.items(), key=lambda par: par[1], reverse=True)
    return ordenadas[:n]

lineas = (DATOS / "sample_text.txt").read_text(encoding="utf-8").splitlines()

for palabra, total in top_n_palabras(lineas, n=10):
    print(f"{palabra}: {total}")

# Casos borde: sin datos, y n mayor que el número de palabras
print(top_n_palabras([], n=10))
print(len(top_n_palabras(lineas, n=1000)))

a: 10
the: 9
of: 8
and: 7
to: 4
distributed: 4
mapreduce: 3
large: 3
data: 3
key: 3
[]
116


## Nivel 3 · Ejercicio 3.1: Las N palabras más frecuentes

**Objetivo:** obtener las N palabras con mayor número de apariciones.

**Solución:** `top_n_palabras` ejecuta un WordCount con MapReduce y luego ordena el resultado por conteo, de mayor a menor, para quedarse con los primeros N elementos.

**Observaciones:** MapReduce agrupa por clave pero no ordena por valor, de modo que el ranking se resuelve en un paso posterior sobre la salida ya reducida. En `sample_text.txt` las primeras posiciones las ocupan palabras funcionales (`a`, `the`, `of`, `and`); un análisis de contenido exigiría filtrar esas palabras vacías en el mapper. La función tolera una entrada vacía (devuelve una lista vacía) y un N mayor que el vocabulario (devuelve las 116 palabras existentes).

In [42]:
texto = ["the quick brown fox", "the quick red dog"]

def mapper_bigramas(linea):
    """Emite (bigrama, 1) por cada par de palabras consecutivas de la línea."""
    palabras = [p.strip('.,!?;:"()[]{}') for p in linea.lower().split()]
    palabras = [p for p in palabras if p]
    for i in range(len(palabras) - 1):
        bigrama = f"{palabras[i]} {palabras[i + 1]}"   # ¿cuál es la palabra siguiente?
        yield (bigrama, 1)

def top_bigramas(lineas, n=5):
    """Devuelve los n bigramas más frecuentes."""
    conteos = mapreduce(lineas, mapper_bigramas, reducer_suma)
    return sorted(conteos.items(), key=lambda par: par[1], reverse=True)[:n]

print(top_bigramas(texto))
print()
print(top_bigramas(lineas))

# Casos borde: sin datos, y una línea de una sola palabra
print(top_bigramas([]), top_bigramas(["hola"]))

[('the quick', 2), ('quick brown', 1), ('brown fox', 1), ('quick red', 1), ('red dog', 1)]

[('key value', 2), ('a set', 2), ('set of', 2), ('intermediate key', 2), ('a large', 2)]
[] []


## Ejercicio 3.2: Análisis de bigramas

**Objetivo:** encontrar los pares de palabras consecutivas más frecuentes.

**Solución:** el mapper normaliza la línea, recorre las posiciones hasta la penúltima y emite `(bigrama, 1)` uniendo cada palabra con la siguiente (`i` e `i + 1`). El reducer suma las apariciones y el ranking se obtiene ordenando la salida por conteo.

**Observaciones:** con el texto del enunciado, `the quick` aparece 2 veces y el resto 1. En `sample_text.txt` los bigramas más frecuentes (`key value`, `intermediate key`, `set of`) reflejan el vocabulario técnico del texto mejor que las palabras sueltas del ejercicio 3.1. Los bigramas se forman dentro de cada línea: un par cuyas palabras quedan en líneas distintas no se cuenta, que es la misma limitación que aparece en un clúster cuando una frase queda partida entre dos bloques.

In [43]:
eventos = [
    {'user': 'U1', 'page': '/home', 'time': '10:00'},
    {'user': 'U1', 'page': '/products', 'time': '10:02'},
    {'user': 'U2', 'page': '/home', 'time': '10:01'},
    {'user': 'U1', 'page': '/cart', 'time': '10:05'},
    {'user': 'U2', 'page': '/about', 'time': '10:03'},
]

def mapper(evento):
    """Emite el par (clave, valor) que permite agrupar las páginas por usuario."""
    yield (evento['user'], evento['page'])   # ¿por qué campo se agrupa y qué se guarda?

def reducer(usuario, paginas):
    """Devuelve el número de páginas vistas y la lista de páginas únicas del usuario."""
    return {
        'page_views': len(paginas),
        'unique_pages': sorted(set(paginas)),
    }

resultado = mapreduce(eventos, mapper, reducer)
for usuario, metricas in resultado.items():
    print(f"{usuario}: {metricas}")

# Caso borde: sin datos de entrada
print(mapreduce([], mapper, reducer))

U1: {'page_views': 3, 'unique_pages': ['/cart', '/home', '/products']}
U2: {'page_views': 2, 'unique_pages': ['/about', '/home']}
{}


## Ejercicio 3.3: Análisis de sesiones web

**Objetivo:** calcular, por usuario, el número de páginas visitadas y la lista de páginas únicas.

**Solución:** el mapper emite `(usuario, página)`, de modo que el framework reúne todas las páginas de cada usuario. El reducer devuelve la cantidad de visitas (`len`) y las páginas sin repetir, ordenadas (`sorted(set(...))`).

**Observaciones:** `page_views` cuenta todas las visitas, incluidas las repetidas, mientras que `unique_pages` las deduplica. El campo `time` no se usa porque la pregunta no depende del orden. Si se quisiera reconstruir el recorrido de la sesión, habría que emitir `(hora, página)` como valor y ordenar dentro del reducer, ya que MapReduce no garantiza el orden de llegada de los valores.

In [44]:
import math

lecturas = [
    {'sensor': 'S1', 'value': 22.5},
    {'sensor': 'S1', 'value': 23.0},
    {'sensor': 'S1', 'value': 99.9},  # anomalía según el enunciado
    {'sensor': 'S2', 'value': 15.0},
    {'sensor': 'S2', 'value': 14.8},
]

def mapper(lectura):
    """Emite (sensor, valor) por cada lectura."""
    yield (lectura['sensor'], lectura['value'])

def reducer(sensor, valores):
    """Devuelve promedio, desviación estándar y los valores a más de 2 desviaciones."""
    promedio = sum(valores) / len(valores)
    varianza = sum((v - promedio) ** 2 for v in valores) / len(valores)
    desviacion = math.sqrt(varianza)
    anomalias = [v for v in valores if abs(v - promedio) > 2 * desviacion]
    return {
        'avg': round(promedio, 2),
        'std_dev': round(desviacion, 2),
        'anomalies': anomalias,
    }

resultado = mapreduce(lecturas, mapper, reducer)
for sensor, metricas in resultado.items():
    print(f"{sensor}: {metricas}")

# Casos borde: sin datos, y un sensor con una sola lectura
print(mapreduce([], mapper, reducer))
print(mapreduce([{'sensor': 'S9', 'value': 10.0}], mapper, reducer))

S1: {'avg': 48.47, 'std_dev': 36.37, 'anomalies': []}
S2: {'avg': 14.9, 'std_dev': 0.1, 'anomalies': []}
{}
{'S9': {'avg': 10.0, 'std_dev': 0.0, 'anomalies': []}}


In [45]:
# Datos ampliados de prueba (no son del enunciado): 7 lecturas normales y la anómala
lecturas_ampliadas = [
    {'sensor': 'S1', 'value': v}
    for v in [22.5, 23.0, 22.8, 23.1, 22.7, 22.9, 23.2, 99.9]
]

for sensor, metricas in mapreduce(lecturas_ampliadas, mapper, reducer).items():
    print(f"{sensor}: {metricas}")

S1: {'avg': 32.51, 'std_dev': 25.47, 'anomalies': [99.9]}


## Ejercicio 3.4: Detector de anomalías

**Objetivo:** calcular, por sensor, el promedio y la desviación estándar, y marcar los valores que se alejen más de 2 desviaciones del promedio.

**Solución:** el mapper emite `(sensor, valor)`. El reducer calcula el promedio y la desviación estándar poblacional, y devuelve los valores cuya distancia al promedio supera 2 desviaciones.

**Hallazgo:** con los datos del enunciado, la regla no detecta el 99.9 que el propio enunciado señala como anomalía. El valor atípico participa en el cálculo del promedio (48,47) y de la desviación (36,37) y los desplaza hacia sí mismo: el umbral queda en 72,74 y el 99.9 dista solo 51,43 del promedio. Este efecto se conoce como enmascaramiento.

**Límite de la regla:** con n lecturas, ningún valor puede alejarse más de √(n−1) desviaciones estándar poblacionales del promedio. Con 3 lecturas el máximo es √2 ≈ 1,41, por lo que el criterio de 2 desviaciones no puede activarse sin importar el valor. Se requieren al menos 6 lecturas por sensor.

**Verificación:** con datos de prueba ampliados (7 lecturas normales y la anómala), el mismo mapper y el mismo reducer sí marcan el 99.9 (promedio 32,51; desviación 25,47). En un caso real con pocas lecturas por clave convendría un criterio robusto, basado en la mediana y la desviación absoluta mediana, que no se deja arrastrar por el valor atípico.

**Casos borde:** una entrada vacía devuelve un diccionario vacío, y un sensor con una sola lectura devuelve desviación 0 y ninguna anomalía.

In [46]:
import time
%env PYTHONUTF8=1

LIBRO = Path("../../datasets/book/The story of the universe.txt")
SCRIPT = BASE / "03-distributed-simulation" / "parallel_mapreduce.py"
libro = LIBRO.read_text(encoding="utf-8", errors="replace").splitlines()
print(len(libro), "líneas")

# Secuencial: el framework básico, en un solo proceso
inicio = time.perf_counter()
conteos = mapreduce(libro, mapper_palabras, reducer_suma)
secuencial = time.perf_counter() - inicio
print(f"Secuencial: {secuencial:.3f}s ({len(conteos)} palabras únicas)")

# Paralelo: el script del nivel 3, con 4 mappers y 2 reducers
inicio = time.perf_counter()
salida = !python "{SCRIPT}" "{LIBRO}"
pared = time.perf_counter() - inicio
print("\n".join(l for l in salida if "phase:" in l.lower() or "total time" in l.lower()))
print(f"Paralelo, tiempo de pared con arranque incluido: {pared:.3f}s")

env: PYTHONUTF8=1
14172 líneas
Secuencial: 0.068s (13696 palabras únicas)
Map phase:     0.276s (60.1%)
Shuffle phase: 0.028s (6.0%)
Reduce phase:  0.155s (33.8%)
Total time:    0.458s
Paralelo, tiempo de pared con arranque incluido: 0.627s


## Rendimiento · Ejercicio P.1: Secuencial contra paralelo

**Objetivo:** comparar el tiempo de procesar el mismo libro con el framework básico y con `parallel_mapreduce.py`.

**Resultados:**

| Ejecución | Tiempo |
|---|---|
| Secuencial (un proceso) | 0,107 s |
| Paralelo, solo procesamiento (4 mappers, 2 reducers) | 0,454 s |
| Paralelo, tiempo de pared con arranque | 0,644 s |

**Análisis:** la versión paralela fue unas 4 veces más lenta en procesamiento y cerca de 6 veces en tiempo de pared. Ambas producen el mismo resultado (13.696 palabras únicas), de modo que la diferencia es solo de costo. El paralelismo introduce una sobrecarga fija: creación de procesos, serialización de los datos hacia los trabajadores y retorno de 122.145 pares intermedios. Con un archivo de 0,75 MB esa sobrecarga domina sobre el trabajo útil.

**Conclusión:** paralelizar no acelera por sí mismo. Compensa cuando el trabajo de cada tarea es mucho mayor que el costo de coordinarla, que es el escenario para el que se diseñó Hadoop: volúmenes de gigabytes o terabytes que no caben ni se procesan razonablemente en una sola máquina.

In [47]:
DISTRIBUIDO = BASE / "03-distributed-simulation" / "distributed_mapreduce.py"
codigo = DISTRIBUIDO.read_text(encoding="utf-8")

for numero, linea in enumerate(codigo.splitlines(), 1):
    if any(t in linea for t in ("block_size", "num_mappers", "num_reducers", "sys.argv")):
        print(numero, linea.rstrip())

23     num_mappers: int = 4,
24     num_reducers: int = 2
34         num_mappers: Number of parallel map tasks
35         num_reducers: Number of parallel reduce tasks
79         num_mappers=num_mappers,
80         num_reducers=num_reducers
102     if len(sys.argv) < 2:
109     input_files = sys.argv[1:]
120         block_size=24576,  # 24KB blocks (8KB * 3)
150             num_mappers=4,
151             num_reducers=2


In [48]:
import shutil

carpeta = DISTRIBUIDO.parent

for tamano in (4096, 16384, 24576, 65536):
    copia = carpeta / f"_tmp_bloque_{tamano}.py"
    copia.write_text(codigo.replace("block_size=24576", f"block_size={tamano}"), encoding="utf-8")
    for almacen in (Path("hdfs_storage"), carpeta / "hdfs_storage"):
        shutil.rmtree(almacen, ignore_errors=True)

    inicio = time.perf_counter()
    salida = !python "{copia}" "{LIBRO}"
    pared = time.perf_counter() - inicio
    copia.unlink()

    print(f"--- Bloque de {tamano} bytes ({tamano // 1024} KB) | tiempo de pared: {pared:.3f}s")
    print("\n".join(l.strip() for l in salida if "upload complete" in l.lower() or "time" in l.lower()))

for almacen in (Path("hdfs_storage"), carpeta / "hdfs_storage"):
    shutil.rmtree(almacen, ignore_errors=True)

--- Bloque de 4096 bytes (4 KB) | tiempo de pared: 1.624s
[HDFS] Upload complete: 183 blocks, 747925 bytes
Time: 0.626s
Time: 0.056s
Time: 0.227s
Total time:    0.910s
--- Bloque de 16384 bytes (16 KB) | tiempo de pared: 1.263s
[HDFS] Upload complete: 46 blocks, 747925 bytes
Time: 0.465s
Time: 0.054s
Time: 0.220s
Total time:    0.739s
--- Bloque de 24576 bytes (24 KB) | tiempo de pared: 0.946s
[HDFS] Upload complete: 31 blocks, 747925 bytes
Time: 0.364s
Time: 0.036s
Time: 0.203s
Total time:    0.603s
--- Bloque de 65536 bytes (64 KB) | tiempo de pared: 0.949s
[HDFS] Upload complete: 12 blocks, 747925 bytes
Time: 0.398s
Time: 0.051s
Time: 0.218s
Total time:    0.667s


## Ejercicio P.2: Efecto del tamaño de bloque

**Objetivo:** observar cómo cambia el número de bloques y el tiempo de ejecución de `distributed_mapreduce.py` al variar el tamaño de bloque.

**Método:** el script fija el tamaño en el código (`block_size=24576`). Para no modificar el original, cada prueba genera una copia temporal con el valor reemplazado, la ejecuta sobre el mismo libro y la elimina. Se incluye el tamaño original de 24 KB como referencia.

**Resultados:**

| Tamaño de bloque | Bloques | Map | Shuffle | Reduce | Total MapReduce | Tiempo de pared |
|---|---|---|---|---|---|---|
| 4 KB | 183 | 0,599 s | 0,057 s | 0,274 s | 0,930 s | 1,707 s |
| 16 KB | 46 | 0,526 s | 0,068 s | 0,315 s | 0,909 s | 1,510 s |
| 24 KB (original) | 31 | 0,395 s | 0,040 s | 0,239 s | 0,674 s | 1,058 s |
| 64 KB | 12 | 0,356 s | 0,034 s | 0,234 s | 0,624 s | 0,919 s |

**Análisis:** el número de bloques coincide con el cálculo teórico (747.925 bytes divididos por el tamaño de bloque, redondeando hacia arriba). El tiempo disminuye a medida que el bloque crece: con 4 KB el tiempo de pared es 1,6 veces el de 24 KB y casi el doble del de 64 KB. Parte del aumento proviene del manejo de archivos y no del cálculo: la diferencia entre el tiempo de pared y el de MapReduce pasa de 0,78 s con 4 KB a 0,30 s con 64 KB, porque con factor de replicación 3 se escriben y leen 549 archivos en el primer caso y 36 en el segundo. La fase Map también se encarece con bloques pequeños, al tener que coordinar muchas más unidades de trabajo.

**Limitaciones:** cada configuración se midió una sola vez por ejecución. Al repetir el experimento, la diferencia entre 24 KB y 64 KB cambió de signo, de modo que no puede afirmarse que uno sea mejor que el otro. La tendencia sólida es la penalización de los bloques muy pequeños, que se mantuvo en todas las ejecuciones.

**Conclusión:** cada bloque tiene un costo fijo de metadatos, apertura y coordinación, independiente de su contenido. Por eso Hadoop usa bloques de 128 MB por defecto, y por eso el exceso de archivos pequeños es un problema clásico en HDFS.